# Kombinle parça haritası eğitimi

Bu defter, boydan (üzerinde, aynada) çekilmiş fotoğrafta **hangi parçanın nerede olduğunu** piksel piksel bulan
küçük bir model eğitir: üst, ceket/mont, pantolon/şort, etek, elbise/tulum, ayakkabı, çanta, şapka, gözlük,
atkı/eşarp, kemer, çorap.

Uygulama bu haritayla boydan fotoğraftaki parçaları **internetsiz** ayırır, kullanıcıya "hangilerini ekleyeyim?" diye
sorar ve seçilen parçayı fotoğraftan tam yerinden keser. Yapay zekâ bulutuna gitmeye gerek kalmaz.

**Nasıl çalıştırılır?** (kıyafet tanıma defteriyle aynı)
1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**.
2. **Çalışma zamanı → Tümünü çalıştır**. Drive izni iste­nirse **izin ver** (kopma olursa kaldığı yerden devam eder).
3. Yaklaşık **1,5–2 saat** sürer.
4. Son hücre GitHub anahtarını ister ve modeli Kombinle'ye gönderir.

**Veri:** [Fashionpedia](https://fashionpedia.github.io/home/) giysi maskeleri (CC BY 4.0). Ticari kullanıma kapalı fotoğraflar alınmaz.

In [ ]:
# ---------- Ayarlar ----------
import os, sys, json, re, random, time, glob, math, shutil, subprocess, zipfile
from collections import Counter, defaultdict

REPO = 'aliyuksekal/kombinle'
DAL = 'main'
GIRDI = 320                          # modelin gördüğü kare resim
ADIM = 4                             # harita GIRDI/ADIM kare (320 → 80×80)
DONEM_ISINMA = 4
DONEM_INCE = 22
TOPLU = 16
TOHUM = 11

SINAMA = os.environ.get('KB_SINAMA') == '1'
if SINAMA:
    GIRDI, TOPLU = 128, 8
CIKIS = GIRDI // ADIM
CALISMA = '/content/kb' if os.path.isdir('/content') and not SINAMA else os.path.abspath(os.environ.get('KB_CALISMA', 'kb'))
VERI = os.path.join(CALISMA, 'fashionpedia')     # kıyafet tanıma defteriyle aynı yer: indirildiyse yeniden inmez
HARITA = os.path.join(CALISMA, 'harita')
CIKTI = os.path.join(CALISMA, 'cikti')
for d in (VERI, HARITA, CIKTI):
    os.makedirs(d, exist_ok=True)
random.seed(TOHUM)
print('Çalışma klasörü:', CALISMA, '(sınama)' if SINAMA else '')

# Google Drive: eğitim her dönem sonunda buraya kaydedilir. Colab bağlantısı koparsa
# "Tümünü çalıştır" ile yeniden başlatınca kaldığı yerden devam eder.
KALICI = None
if os.path.isdir('/content') and not SINAMA:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        KALICI = '/content/drive/MyDrive/kombinle_egitim'
    except Exception as e:
        print('Drive bağlanmadı (sorun değil, ara kayıtlar yalnız bu oturumda tutulur):', e)
KALICI = KALICI or os.path.join(CALISMA, 'kalici')
os.makedirs(KALICI, exist_ok=True)
print('Ara kayıtlar:', KALICI)

In [ ]:
# ---------- Ekran kartı kontrolü ----------
try:
    import tensorflow as tf
    import keras
    gpu = tf.config.list_physical_devices('GPU')
    print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', gpu[0].name if gpu else 'YOK')
    if gpu:
        keras.mixed_precision.set_global_policy('mixed_float16')   # T4'te ~2 kat hızlı; telefona float32 gider
    elif not SINAMA:
        print('\n⚠️  GPU seçilmemiş. Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.\n'
              '   GPU olmadan eğitim çok uzun sürer.')
    if not hasattr(tf, 'lite'):
        print('⚠️  Bu TensorFlow sürümünde tf.lite yok; son adımda telefon modeli üretilemez.')
except ImportError:
    tf = None
    print('TensorFlow yok (yalnız veri hazırlama sınanabilir).')

## 1. Fashionpedia'yı indir
Kıyafet tanıma defterini aynı oturumda çalıştırdıysan yeniden inmez.

In [ ]:
ADRESLER = {
    'train2020.zip': 'https://s3.amazonaws.com/ifashionist-dataset/images/train2020.zip',
    'val_test2020.zip': 'https://s3.amazonaws.com/ifashionist-dataset/images/val_test2020.zip',
    'instances_attributes_train2020.json': 'https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_train2020.json',
    'instances_attributes_val2020.json': 'https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json',
}


def indir(ad, adres, klasor=None):
    hedef = os.path.join(klasor or VERI, ad)
    if os.path.exists(hedef) and os.path.getsize(hedef) > 0:
        print('var:', ad)
        return hedef
    print('indiriliyor:', ad)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', hedef + '.part', adres], check=True)
    os.replace(hedef + '.part', hedef)
    return hedef


def ac(zip_yolu, klasor):
    if os.path.isdir(klasor) and os.listdir(klasor):
        print('açık:', os.path.basename(klasor))
        return
    print('açılıyor:', os.path.basename(zip_yolu))
    with zipfile.ZipFile(zip_yolu) as z:
        z.extractall(VERI)


FP_KATEGORI = ['shirt, blouse', 'top, t-shirt, sweatshirt', 'sweater', 'cardigan', 'jacket', 'vest', 'pants', 'shorts', 'skirt',
               'coat', 'dress', 'jumpsuit', 'cape', 'glasses', 'hat', 'headband, head covering, hair accessory', 'tie', 'glove',
               'watch', 'belt', 'leg warmer', 'tights, stockings', 'sock', 'shoe', 'bag, wallet', 'scarf', 'umbrella', 'hood',
               'collar', 'lapel', 'epaulette', 'sleeve', 'pocket', 'neckline', 'buckle', 'zipper']


def sahte_veri_kur():
    """Geliştirici sınaması: Fashionpedia biçiminde küçük sahte veri (çokgen maskeler, kol parçaları, boy özellikleri)."""
    from PIL import Image, ImageDraw
    oz = [('plain (pattern)', 'textile pattern'), ('stripe', 'textile pattern'), ('check', 'textile pattern'), ('floral', 'textile pattern'),
          ('dot', 'textile pattern'), ('leopard', 'textile pattern'), ('abstract', 'textile pattern'), ('jeans', 'nickname'),
          ('maxi (length)', 'length'), ('mini (length)', 'length'), ('knee (length)', 'length'), ('midi', 'length'),
          ('above-the-knee (length)', 'length'), ('sleeveless', 'length'), ('short (length)', 'length'), ('elbow-length', 'length'),
          ('wrist-length', 'length'), ('hoodie', 'nickname'), ('trench (coat)', 'nickname'), ('blazer', 'nickname'),
          ('leggings', 'nickname'), ('tunic (top)', 'nickname'), ('denim', 'textile material')]
    giysi = [i for i, k in enumerate(FP_KATEGORI) if FP_KATEGORI.index('umbrella') > i]
    kol_kat = FP_KATEGORI.index('sleeve')
    for bolum, klasor, n in (('train2020', 'train', 320), ('val2020', 'test', 80)):
        os.makedirs(os.path.join(VERI, klasor), exist_ok=True)
        resimler, notlar = [], []
        for i in range(n):
            ad = '%s_%05d.jpg' % (bolum, i)
            im = Image.new('RGB', (320, 400), (random.randint(150, 255),) * 3)
            dr = ImageDraw.Draw(im)
            for k in range(3):
                x, y = random.randint(0, 200), random.randint(0, 260)
                w, h = random.randint(50, 120), random.randint(50, 140)
                c = random.choice(giysi)
                renk = tuple((c * 37 + j * 91) % 256 for j in range(3))   # tür → sabit renk: sahte veride öğrenilebilir
                cokgen = [x, y, x + w, y, x + w, y + h, x, y + h]
                dr.polygon(cokgen, fill=renk)
                a = random.sample(range(len(oz)), 3)
                notlar.append({'id': len(notlar) + 1, 'image_id': i + 1, 'category_id': c, 'attribute_ids': a,
                               'bbox': [x, y, w, h], 'area': w * h, 'iscrowd': 0, 'segmentation': [cokgen]})
                if c < 5:   # üst giyimde bir kol parçası
                    kw, kh = max(10, w // 4), max(10, h // 2)
                    notlar.append({'id': len(notlar) + 1, 'image_id': i + 1, 'category_id': kol_kat,
                                   'attribute_ids': [random.choice([13, 14, 15, 16])], 'bbox': [x, y, kw, kh], 'area': kw * kh,
                                   'iscrowd': 0, 'segmentation': [[x, y, x + kw, y, x + kw, y + kh, x, y + kh]]})
            im.save(os.path.join(VERI, klasor, ad), quality=85)
            resimler.append({'id': i + 1, 'file_name': ad, 'width': 320, 'height': 400, 'license': 1 + i % 3})
        js = {'info': {}, 'categories': [{'id': i, 'name': k, 'supercategory': 'x', 'level': 1} for i, k in enumerate(FP_KATEGORI)],
              'attributes': [{'id': i, 'name': a, 'supercategory': s, 'level': 1} for i, (a, s) in enumerate(oz)],
              'licenses': [{'id': 1, 'name': 'Attribution License', 'url': 'http://creativecommons.org/licenses/by/2.0/'},
                           {'id': 2, 'name': 'Attribution-NonCommercial License', 'url': 'http://creativecommons.org/licenses/by-nc/2.0/'},
                           {'id': 3, 'name': 'No known copyright restrictions', 'url': 'http://flickr.com/commons/usage/'}],
              'images': resimler, 'annotations': notlar}
        json.dump(js, open(os.path.join(VERI, 'instances_attributes_%s.json' % bolum), 'w'))


if SINAMA:
    sahte_veri_kur()
else:
    for ad, adres in ADRESLER.items():
        indir(ad, adres)
    ac(os.path.join(VERI, 'train2020.zip'), os.path.join(VERI, 'train'))
    ac(os.path.join(VERI, 'val_test2020.zip'), os.path.join(VERI, 'test'))


def resim_klasoru(bolum):
    """Zip'ten çıkan klasör adını bulur (train / test)."""
    for aday in (['train', 'train2020'] if bolum == 'train2020' else ['test', 'val2020', 'val_test2020']):
        yol = os.path.join(VERI, aday)
        if os.path.isdir(yol):
            return yol
    raise FileNotFoundError('Resim klasörü bulunamadı: ' + bolum)


def lisans_uygun(lisans):
    """Ticari kullanıma kapalı (NonCommercial) fotoğraflar eğitime alınmaz."""
    if not lisans:
        return False
    metin = (lisans.get('name', '') + ' ' + lisans.get('url', '')).lower()
    return not re.search(r'noncommercial|non-commercial|by-nc|/nc', metin)


print({ad: os.path.getsize(os.path.join(VERI, ad)) // 1_000_000 for ad in os.listdir(VERI) if os.path.isfile(os.path.join(VERI, ad))}, 'MB')

## 2. Parça haritalarını çiz
Her fotoğraf beyaz zeminle kareye tamamlanır (uygulamadaki gibi) ve giysi çokgenleri haritaya boyanır.
Büyük parçalar önce, küçükler (kemer, çanta) üstüne çizilir. Yaka, cep, kol gibi giysi *bölümleri* ayrı parça sayılmaz.

In [ ]:
from PIL import Image, ImageDraw
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

# (id, uygulamadaki grup, Fashionpedia kategori adı başları, telefonda en az alan oranı)
PARCALAR = [
    ('zemin', '', (), 0),
    ('ust', 'ust', ('shirt', 'top', 'sweater'), 0.03),
    ('dis', 'dis', ('cardigan', 'jacket', 'vest', 'coat', 'cape'), 0.03),
    ('alt', 'alt', ('pants', 'shorts'), 0.03),
    ('etek', 'alt', ('skirt',), 0.03),
    ('elbise', 'tek', ('dress', 'jumpsuit'), 0.04),
    ('ayakkabi', 'ayk', ('shoe',), 0.004),
    ('canta', 'aks', ('bag',), 0.008),
    ('sapka', 'aks', ('hat', 'headband'), 0.004),
    ('gozluk', 'aks', ('glasses',), 0.002),
    ('atki', 'aks', ('scarf',), 0.008),
    ('kemer', 'aks', ('belt',), 0.002),
    ('corap', 'ic', ('sock', 'tights', 'leg warmer'), 0.004),
]
PARCA_NO = {p[0]: i for i, p in enumerate(PARCALAR)}


def parca_no(kat):
    k = kat.lower()
    for i, p in enumerate(PARCALAR):
        if p[2] and k.startswith(p[2]):
            return i
    return 0


try:
    from pycocotools import mask as coco_maske
except Exception:
    coco_maske = None


def harita_ciz(is_):
    yol, hedef_resim, hedef_harita, notlar = is_
    try:
        with Image.open(yol) as im:
            im = im.convert('RGB')
            W, H = im.size
            m = Image.new('L', (W, H), 0)
            dr = ImageDraw.Draw(m)
            for no, seg, _ in sorted(notlar, key=lambda n: -n[2]):
                if isinstance(seg, list):
                    for c in seg:
                        if len(c) >= 6:
                            dr.polygon([(c[i], c[i + 1]) for i in range(0, len(c) - 1, 2)], fill=no)
                elif isinstance(seg, dict) and coco_maske is not None:
                    rle = seg if isinstance(seg.get('counts'), (bytes, str)) else coco_maske.frPyObjects(seg, H, W)
                    ikili = coco_maske.decode(rle)
                    m.paste(no, mask=Image.fromarray((ikili > 0).astype('uint8') * 255))
                else:
                    return None
            o = GIRDI / max(W, H)
            w, h = max(1, round(W * o)), max(1, round(H * o))
            x, y = (GIRDI - w) // 2, (GIRDI - h) // 2
            kare = Image.new('RGB', (GIRDI, GIRDI), (255, 255, 255))
            kare.paste(im.resize((w, h), Image.BILINEAR), (x, y))
            kh = Image.new('L', (GIRDI, GIRDI), 0)
            kh.paste(m.resize((w, h), Image.NEAREST), (x, y))
            kare.save(hedef_resim, quality=90)
            kh.save(hedef_harita)
            return hedef_resim
    except Exception as e:
        print('atlandı:', os.path.basename(yol), e)
        return None


def haritalari_hazirla(bolum, sinir=None):
    js = json.load(open(os.path.join(VERI, 'instances_attributes_%s.json' % bolum)))
    kat = {c['id']: c['name'] for c in js['categories']}
    lis = {l['id']: l for l in js.get('licenses', [])}
    resim = {r['id']: r for r in js['images']}
    notlar = defaultdict(list)
    for n in js['annotations']:
        no = parca_no(kat.get(n['category_id'], ''))
        if no:
            notlar[n['image_id']].append((no, n.get('segmentation'), n.get('area', 0)))
    klasor = resim_klasoru(bolum)
    hedef = os.path.join(HARITA, bolum)
    os.makedirs(hedef, exist_ok=True)
    isler, hazir = [], []
    kimlikler = [r for r in notlar if r in resim and lisans_uygun(lis.get(resim[r].get('license')))]
    random.shuffle(kimlikler)
    for rid in kimlikler[:sinir]:
        r = resim[rid]
        a = os.path.join(hedef, '%d.jpg' % rid)
        b = os.path.join(hedef, '%d.png' % rid)
        if os.path.exists(a) and os.path.exists(b):
            hazir.append((a, b))
        else:
            isler.append((os.path.join(klasor, r['file_name']), a, b, notlar[rid]))
    bas = time.time()
    with ProcessPoolExecutor(max_workers=max(2, os.cpu_count() or 2), mp_context=multiprocessing.get_context('fork')) as h:
        for k, sonuc in enumerate(h.map(harita_ciz, isler, chunksize=16)):
            if sonuc:
                hazir.append((sonuc, sonuc[:-4] + '.png'))
            if k and k % 3000 == 0:
                print('  %d / %d (%.0f sn)' % (k, len(isler), time.time() - bas))
    print(bolum, ':', len(hazir), 'fotoğraf hazır', '' if coco_maske else '(pycocotools yok: RLE maskeli fotoğraflar atlandı)')
    return hazir


egitim_cift = haritalari_hazirla('train2020')
dogrulama_cift = haritalari_hazirla('val2020')

# piksel sıklığına göre ağırlık: küçük parçalar (kemer, gözlük) da öğrenilsin
import numpy as np
sayac = np.zeros(len(PARCALAR))
for _, b in random.sample(egitim_cift, min(2000, len(egitim_cift))):
    sayac += np.bincount(np.asarray(Image.open(b)).ravel(), minlength=len(PARCALAR))[:len(PARCALAR)]
siklik = sayac / max(1, sayac.sum())
orta = np.median(siklik[siklik > 0]) if (siklik > 0).any() else 1.0
AGIRLIKLAR = np.clip(np.sqrt(orta / np.maximum(siklik, 1e-6)), 0.3, 6.0).astype('float32')
for p, s, a in zip(PARCALAR, siklik, AGIRLIKLAR):
    print('  %-9s %%%5.2f piksel  ağırlık %.2f' % (p[0], s * 100, a))

## 3. Modeli eğit
Temel: **MobileNetV3-Large** (telefonda hızlı). Üstüne ayrıntıyı geri getiren bir çözücü eklenir; çıkış 80×80'lik
bir parça haritası. Her dönem sonunda Drive'a kaydedilir.

In [ ]:
if tf is None:
    raise SystemExit('TensorFlow gerekli (Colab\'da hazır gelir).')
from keras import layers, ops

keras.utils.set_random_seed(TOHUM)
K = len(PARCALAR)


def veri_kumesi(ciftler, egitim, toplu=TOPLU):
    a = [c[0] for c in ciftler]
    b = [c[1] for c in ciftler]
    ds = tf.data.Dataset.from_tensor_slices((a, b))
    if egitim:
        ds = ds.shuffle(len(a), seed=TOHUM, reshuffle_each_iteration=True)

    def yukle(ra, rb):
        im = tf.image.convert_image_dtype(tf.io.decode_jpeg(tf.io.read_file(ra), channels=3), tf.float32)
        m = tf.io.decode_png(tf.io.read_file(rb), channels=1)
        if egitim:
            # aynalama ve rastgele yakınlaştırma resim ve haritaya birlikte uygulanır
            ikisi = tf.concat([im, tf.cast(m, tf.float32)], -1)
            ikisi = tf.image.random_flip_left_right(ikisi)
            kenar = tf.cast(tf.cast(GIRDI, tf.float32) * tf.random.uniform([], 0.75, 1.0), tf.int32)
            ikisi = tf.image.random_crop(ikisi, [kenar, kenar, 4])
            im = tf.image.resize(ikisi[..., :3], [GIRDI, GIRDI])
            m = tf.cast(tf.image.resize(ikisi[..., 3:], [GIRDI, GIRDI], method='nearest'), tf.uint8)
            im = tf.image.random_brightness(im, 0.12)
            im = tf.image.random_contrast(im, 0.85, 1.15)
            im = tf.image.random_saturation(im, 0.8, 1.2)
            im = tf.clip_by_value(im, 0.0, 1.0)
        m = tf.image.resize(m, [CIKIS, CIKIS], method='nearest')
        im = tf.ensure_shape(im, [GIRDI, GIRDI, 3])   # model sabit boy bekler (doğrulamada boy bilinmiyor)
        return im * 255.0, tf.ensure_shape(tf.cast(tf.squeeze(m, -1), tf.int32), [CIKIS, CIKIS])

    return ds.map(yukle, num_parallel_calls=tf.data.AUTOTUNE).batch(toplu).prefetch(tf.data.AUTOTUNE)


AGIRLIK_T = tf.constant(AGIRLIKLAR)


def agirlikli_kayip(y_dogru, y_tahmin):
    y = ops.reshape(ops.cast(y_dogru, 'int32'), (-1,))
    p = ops.reshape(ops.cast(y_tahmin, 'float32'), (-1, K))
    k = keras.losses.sparse_categorical_crossentropy(y, p)
    w = ops.take(AGIRLIK_T, y)
    return ops.sum(k * w) / ops.maximum(ops.sum(w), 1.0)


def model_kur(agirlik='imagenet'):
    giris = keras.Input((GIRDI, GIRDI, 3), name='resim')
    taban = keras.applications.MobileNetV3Large(input_tensor=giris, include_top=False, weights=agirlik,
                                                 include_preprocessing=True)

    def ozellik(boyut):
        bulunan = None
        for k in taban.layers:
            try:
                s = k.output.shape
            except Exception:
                continue
            if len(s) == 4 and s[1] == boyut:
                bulunan = k.output
        assert bulunan is not None, boyut
        return bulunan

    f4, f8, f16, f32 = (ozellik(GIRDI // a) for a in (4, 8, 16, 32))

    def blok(x, n):
        x = layers.Conv2D(n, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        return layers.ReLU()(x)

    # LR-ASPP benzeri bağlam: küresel ortalama ile kanal dikkati
    x = blok(f32, 128)
    dikkat = layers.GlobalAveragePooling2D(keepdims=True)(f32)
    dikkat = layers.Conv2D(128, 1, activation='sigmoid')(dikkat)
    x = layers.Multiply()([x, dikkat])
    for f, n in ((f16, 96), (f8, 64), (f4, 48)):
        x = layers.UpSampling2D(2, interpolation='bilinear')(x)
        x = layers.Concatenate()([x, layers.Conv2D(n, 1, use_bias=False)(f)])
        x = blok(x, n)
    cikis = layers.Conv2D(K, 1, activation='softmax', dtype='float32', name='harita')(x)
    return keras.Model(giris, cikis), taban


egitim_ds = veri_kumesi(egitim_cift, True)
dogrulama_ds = veri_kumesi(dogrulama_cift, False)
model, taban = model_kur()
print('Çıkış:', model.output.shape)

DURUM = os.path.join(KALICI, 'parca_durum.json')
AGIRLIK = os.path.join(KALICI, 'parca_son.weights.h5')
EN_IYI = os.path.join(KALICI, 'parca_en_iyi.weights.h5')
IMZA = {'parcalar': [p[0] for p in PARCALAR], 'girdi': GIRDI, 'adim': ADIM, 'sinama': SINAMA}
durum = {'asama': 0, 'donem': 0}
if os.path.exists(DURUM) and os.path.exists(AGIRLIK):
    try:
        d = json.load(open(DURUM))
        if d.get('imza') == IMZA:
            model.load_weights(AGIRLIK)
            durum = d
            print('Kaldığı yerden devam: %d. aşama, %d. dönem' % (durum['asama'], durum['donem']))
    except Exception as e:
        print('Önceki kayıt okunamadı, baştan başlanıyor:', e)


class DriveKaydi(keras.callbacks.Callback):
    def __init__(self, asama):
        super().__init__()
        self.asama = asama

    def on_epoch_end(self, donem, logs=None):
        self.model.save_weights(AGIRLIK)
        json.dump({'asama': self.asama, 'donem': donem + 1, 'imza': IMZA}, open(DURUM, 'w'))


def derle(oran):
    model.compile(optimizer=keras.optimizers.Adam(oran), loss=agirlikli_kayip,
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name='piksel')])


ISINMA = 1 if SINAMA else DONEM_ISINMA
INCE = 1 if SINAMA else DONEM_INCE
ayrinti = 2 if SINAMA else 1
for k in taban.layers:
    k.trainable = False
if durum['asama'] <= 1 and not (durum['asama'] == 1 and durum['donem'] >= ISINMA):
    print('1. aşama: çözücü')
    derle(1e-3)
    model.fit(egitim_ds, validation_data=dogrulama_ds, epochs=ISINMA, initial_epoch=durum['donem'] if durum['asama'] == 1 else 0,
              verbose=ayrinti, callbacks=[DriveKaydi(1)])
    durum = {'asama': 2, 'donem': 0}

print('2. aşama: tamamı')
for k in taban.layers:
    k.trainable = not isinstance(k, layers.BatchNormalization)
derle(3e-4)
basla = durum['donem'] if durum['asama'] == 2 else 0
if basla < INCE:
    model.fit(egitim_ds, validation_data=dogrulama_ds, epochs=INCE, initial_epoch=basla, verbose=ayrinti,
              callbacks=[DriveKaydi(2),
                         keras.callbacks.ModelCheckpoint(EN_IYI, monitor='val_loss', save_best_only=True, save_weights_only=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
                         keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)])
if os.path.exists(EN_IYI):
    model.load_weights(EN_IYI)

## 4. Ne kadar doğru?
İki ölçü: her parçanın piksel örtüşmesi (IoU) ve uygulamanın asıl işi olan **"parçanın kutusunu doğru buldu mu?"**
(gerçek kutuyla en az %50 örtüşen kutu).

In [ ]:
def kutular(harita, olasilik=None):
    """Telefondaki hesabın aynısı: her parça için bağlı bölgeler, büyüklerin birleşik kutusu."""
    from scipy import ndimage
    sonuc = {}
    for k in range(1, K):
        etk, n = ndimage.label(harita == k)
        if not n:
            continue
        alanlar = ndimage.sum(np.ones_like(etk), etk, range(1, n + 1))
        en = alanlar.max()
        sec = [i + 1 for i, a in enumerate(alanlar) if a >= max(2, en * 0.25)]
        m = np.isin(etk, sec)
        oran = m.sum() / harita.size
        if oran < PARCALAR[k][3]:
            continue
        ys, xs = np.nonzero(m)
        sonuc[k] = (ys.min(), xs.min(), ys.max() + 1, xs.max() + 1)
    return sonuc


def kutu_ortusme(a, b):
    y0, x0, y1, x1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    k = max(0, y1 - y0) * max(0, x1 - x0)
    return k / max(1e-9, (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - k)


kesisim, birlesim = np.zeros(K), np.zeros(K)
bulundu, gercek_say, tahmin_say, dogru_tahmin = Counter(), Counter(), Counter(), Counter()
for im, m in dogrulama_ds:
    t = model.predict(im, verbose=0).argmax(-1)
    m = m.numpy()
    for k in range(K):
        kesisim[k] += np.sum((t == k) & (m == k))
        birlesim[k] += np.sum((t == k) | (m == k))
    for ti, mi in zip(t, m):
        g, p = kutular(mi), kutular(ti)
        for k, kg in g.items():
            gercek_say[k] += 1
            if k in p and kutu_ortusme(kg, p[k]) >= 0.5:
                bulundu[k] += 1
        for k, kp in p.items():
            tahmin_say[k] += 1
            if k in g and kutu_ortusme(g[k], kp) >= 0.5:
                dogru_tahmin[k] += 1
iou = kesisim / np.maximum(birlesim, 1)
dogruluk = {}
print('%-9s %6s %9s %9s' % ('parça', 'IoU', 'buldu', 'isabet'))
for k in range(1, K):
    b = bulundu[k] / max(1, gercek_say[k])
    i = dogru_tahmin[k] / max(1, tahmin_say[k])
    dogruluk[PARCALAR[k][0]] = {'iou': round(float(iou[k]), 3), 'buldu': round(b, 3), 'isabet': round(i, 3), 'ornek': gercek_say[k]}
    print('%-9s %%%5.1f %%%7.1f %%%7.1f  (%d)' % (PARCALAR[k][0], iou[k] * 100, b * 100, i * 100, gercek_say[k]))
ort_iou = float(iou[1:].mean())
print('Ortalama IoU: %%%.1f' % (ort_iou * 100))

## 5. Telefon modeline dönüştür

In [ ]:
keras.mixed_precision.set_global_policy('float32')
model32, _ = model_kur(agirlik=None)
model32.set_weights(model.get_weights())

kayit_yolu = os.path.join(CALISMA, 'kayitli_parca')
shutil.rmtree(kayit_yolu, ignore_errors=True)
model32.export(kayit_yolu)
donustur = tf.lite.TFLiteConverter.from_saved_model(kayit_yolu)
donustur.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = donustur.convert()
MODEL_YOLU = os.path.join(CIKTI, 'kombinle_parca.tflite')
open(MODEL_YOLU, 'wb').write(tflite)
print('Model boyutu: %.1f MB' % (len(tflite) / 1e6))

yorumlayici = tf.lite.Interpreter(model_content=tflite)
yorumlayici.allocate_tensors()
giris_d = yorumlayici.get_input_details()[0]
cikis_d = yorumlayici.get_output_details()
assert list(giris_d['shape']) == [1, GIRDI, GIRDI, 3] and giris_d['dtype'] == np.float32, giris_d
assert len(cikis_d) == 1 and list(cikis_d[0]['shape']) == [1, CIKIS, CIKIS, K], cikis_d
uyum = []
toplu = next(iter(veri_kumesi(dogrulama_cift[:50], False, 50)))[0].numpy()
keras_t = model32.predict(toplu, verbose=0).argmax(-1)
for i in range(len(toplu)):
    yorumlayici.set_tensor(giris_d['index'], toplu[i:i + 1])
    yorumlayici.invoke()
    uyum.append(float((yorumlayici.get_tensor(cikis_d[0]['index'])[0].argmax(-1) == keras_t[i]).mean()))
print('Telefon modeli ile eğitim modelinin piksel uyumu: %%%.1f' % (np.mean(uyum) * 100))
assert np.mean(uyum) >= 0.95, 'Dönüştürmede sorun var; modeli göndermeyin.'

bilgi = {
    'surum': int(time.time()),
    'girdi': GIRDI,
    'cikis': CIKIS,
    'olcek': '0-255',
    'zemin': '#ffffff',
    'siniflar': [{'id': p[0], 'grup': p[1], 'en_az': p[3]} for p in PARCALAR],
    'dogruluk': {'ort_iou': round(ort_iou, 3), 'parca': dogruluk},
    'kaynak': 'Fashionpedia (CC BY 4.0)',
}
BILGI_YOLU = os.path.join(CIKTI, 'kombinle_parca.json')
json.dump(bilgi, open(BILGI_YOLU, 'w'), ensure_ascii=False, indent=1)
print('Kaydedildi:', MODEL_YOLU, BILGI_YOLU)

## Modeli Kombinle'ye gönder
Bir **GitHub anahtarı** gerekiyor (bir kez):
1. GitHub → sağ üst fotoğraf → **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**
2. *Repository access*: **Only select repositories → kombinle**
3. *Permissions → Repository permissions → Contents*: **Read and write**
4. Oluşan anahtarı kopyala, aşağıdaki kutuya yapıştır (ekranda görünmez, hiçbir yere kaydedilmez).

Dosyalar tek bir kayıtla `app/src/main/assets/` klasörüne gider; GitHub yeni APK'yı ve AAB'yi kendisi derler (5–6 dakika).
Anahtarla uğraşmak istemezsen alttaki hücrede dosyaları indirip GitHub'da elle de yükleyebilirsin.

In [ ]:
import base64, getpass, urllib.request


def github(yontem, yol, anahtar, govde=None):
    istek = urllib.request.Request('https://api.github.com/repos/%s%s' % (REPO, yol), method=yontem,
                                   data=json.dumps(govde).encode() if govde is not None else None,
                                   headers={'Authorization': 'Bearer ' + anahtar, 'Accept': 'application/vnd.github+json',
                                            'X-GitHub-Api-Version': '2022-11-28', 'User-Agent': 'kombinle-egitim'})
    with urllib.request.urlopen(istek, timeout=300) as y:
        return json.loads(y.read() or b'{}')


def gonder(dosyalar, mesaj):
    anahtar = getpass.getpass('GitHub anahtarı: ').strip()
    if not anahtar:
        print('Anahtar girilmedi; gönderilmedi.')
        return
    son = github('GET', '/git/ref/heads/' + DAL, anahtar)['object']['sha']
    agac = github('GET', '/git/commits/' + son, anahtar)['tree']['sha']
    ogeler = []
    for yerel, uzak in dosyalar:
        blob = github('POST', '/git/blobs', anahtar, {'content': base64.b64encode(open(yerel, 'rb').read()).decode(), 'encoding': 'base64'})
        ogeler.append({'path': uzak, 'mode': '100644', 'type': 'blob', 'sha': blob['sha']})
    yeni_agac = github('POST', '/git/trees', anahtar, {'base_tree': agac, 'tree': ogeler})['sha']
    kayit = github('POST', '/git/commits', anahtar, {'message': mesaj, 'tree': yeni_agac, 'parents': [son]})['sha']
    github('PATCH', '/git/refs/heads/' + DAL, anahtar, {'sha': kayit})
    print('Gönderildi ✓ GitHub yeni APK\'yı derliyor: https://github.com/%s/actions' % REPO)


if SINAMA:
    print('sınama: gönderim atlandı')
else:
    gonder([(MODEL_YOLU, 'app/src/main/assets/kombinle_parca.tflite'), (BILGI_YOLU, 'app/src/main/assets/kombinle_parca.json')],
           'Kombinle parça haritası: ortalama IoU %%%.0f' % (ort_iou * 100))

In [ ]:
if not SINAMA:
    try:
        from google.colab import files
        files.download(MODEL_YOLU)
        files.download(BILGI_YOLU)
    except Exception as e:
        print('İndirme yalnız Colab\'da çalışır:', e)